In [1]:
import sys
print(sys.executable)

/Users/dchang/Projects/b2b_funnel_mock_data/.venv/bin/python3.13


In [2]:
import duckdb
import pandas as pd

DB_PATH = "/Users/dchang/Projects/b2b_funnel_mock_data/warehouse.duckdb"

con = duckdb.connect(DB_PATH, read_only=True)
accounts = con.execute("SELECT * FROM analytics.nurture_experiment_accounts").df()
con.close()

accounts.head()

,experiment_id,experiment_name,account_id,assigned_group,assigned_at,campaign_id,segment,trigger_lead_id,eligible_leads,first_sql_at,converted_within_60d
0,EXP-2025-07-NURTURE,MQL Nurture Program holdout,0015Ak0000000000BQ,holdout,2025-07-01 07:00:48,7015Ak00000000002U,SMB,00Q5Ak0000000008QM,1,2025-07-07 09:37:13,1
1,EXP-2025-07-NURTURE,MQL Nurture Program holdout,0015Ak0000000002UB,holdout,2025-07-01 08:43:41,7015Ak00000000002U,Mid-Market,00Q5Ak0000000008R0,6,2025-09-11 11:51:59,1
2,EXP-2025-07-NURTURE,MQL Nurture Program holdout,0015Ak00000000016A,treatment,2025-07-01 06:49:29,7015Ak00000000002U,SMB,00Q5Ak00000000091P,2,2025-08-11 09:00:27,1
3,EXP-2025-07-NURTURE,MQL Nurture Program holdout,0015Ak0000000003SO,treatment,2025-07-02 07:33:16,7015Ak00000000002U,Mid-Market,00Q5Ak0000000008JT,1,2025-07-06 10:13:22,1
4,EXP-2025-07-NURTURE,MQL Nurture Program holdout,0015Ak0000000004GH,treatment,2025-07-02 07:49:38,7015Ak00000000002U,Mid-Market,00Q5Ak00000000095S,2,2025-07-24 10:57:18,1


In [3]:
accounts['account_id'].count()

np.int64(2984)

In [4]:
accounts.groupby('assigned_group')['account_id'].count()

assigned_group
holdout       597
treatment    2387
Name: account_id, dtype: int64

In [5]:
accounts.groupby('assigned_group')['account_id'].size()

assigned_group
holdout       597
treatment    2387
Name: account_id, dtype: int64

In [6]:
summary = accounts.groupby('assigned_group').agg(
    accounts=('account_id', 'count'),
    avg_eligible_leads=('eligible_leads', 'mean'),
)
summary['share'] = summary['accounts'] / summary['accounts'].sum()

segment_mix = pd.crosstab(accounts['assigned_group'], accounts['segment'], normalize='index')

display(summary)
print()
display(segment_mix)
print()


,accounts,avg_eligible_leads,share
assigned_group,,,
holdout,597,2.023451,0.200067
treatment,2387,2.088814,0.799933


segment,Enterprise,Mid-Market,SMB
assigned_group,,,
holdout,0.190955,0.391960,0.417085
treatment,0.191873,0.391286,0.416841
